In [1]:
import platform
import numpy as np
import pandas as pd
import h5py
from pathlib import Path

from vascular_superenhancement.utils.path_config import load_path_config, _PROJECT_ROOT

config_name = "local_mac" if platform.system() == "Darwin" else "all_patients"
pc = load_path_config(config_name)

SPLITS_PATH = _PROJECT_ROOT / "splits" / "splits_01-15-26.csv"
splits_df = pd.read_csv(SPLITS_PATH)
test_patients = sorted(splits_df.loc[splits_df["split"] == "test", "patient_id"].tolist())
print(f"Number of test patients: {len(test_patients)}")
print(f"Test patients: {test_patients}")

Found project root at: /home/ayeluru/vascular-superenhancement-4d-flow
Number of test patients: 28
Test patients: ['Balboloop', 'Biswifo', 'Bomatog', 'Boochuto', 'Boumorim', 'Bovutou', 'Cadotueg', 'Detodu', 'Diecudey', 'Diepami', 'Diequipi', 'Dithigog', 'Dublafer', 'Dujomal', 'Elagieg', 'Golotag', 'Grequafie', 'Gueshifa', 'Kuquelok', 'Oduskueb', 'Quetode', 'Runusath', 'Sepigoo', 'Stonscuetof', 'Suquepog', 'Tercippun', 'Tiepolem', 'Tisupey']


In [2]:
import numpy
print(numpy.__version__)

1.26.4


## Inspect HDF5 structure

In [ ]:
print(pc.repository_root)

/home/ayeluru/mnt/fourier/repository/vascular-superenhancement-4d-flow/all_patients


In [3]:
hdf5_path = _PROJECT_ROOT / "working_dir" / "cardiac_processed_cnn_corrected.hdf5"
data = h5py.File(hdf5_path, "r")

print(data.keys())

<KeysViewHDF5 ['Balboloop', 'Biswifo', 'Bomatog', 'Boochuto', 'Boumorim', 'Bovutou', 'Cadotueg', 'Detodu', 'Diecudey', 'Diepami', 'Diequipi', 'Dithigog', 'Dublafer', 'Dujomal', 'Elagieg', 'Golotag', 'Grequafie', 'Gueshifa', 'Kuquelok', 'Oduskueb', 'Quetode', 'Runusath', 'Sepigoo', 'Stonscuetof', 'Suquepog', 'Tercippun', 'Tiepolem', 'Tisupey']>


In [ ]:
data['Balboloop'].shape

(256, 256, 140, 20, 3)

## Compare shapes: HDF5 vs corrected velocity .npy for test patients

In [ ]:
from tqdm import tqdm

corr_vel_dir = pc.repository_root / "corrected_velocities"

rows = []
with h5py.File(hdf5_path, "r") as f:
    for i, pid in enumerate(test_patients):
        print(f"[{i+1}/{len(test_patients)}] {pid}...", flush=True)
        row = {"patient_id": pid}

        if pid in f:
            row["hdf5_shape"] = f[pid].shape
            row["hdf5_dtype"] = str(f[pid].dtype)
        else:
            row["hdf5_shape"] = "NOT FOUND"
            row["hdf5_dtype"] = ""

        npy_path = corr_vel_dir / f"{pid}.npy"
        if npy_path.exists():
            arr = np.load(npy_path)
            row["npy_shape"] = arr.shape
            del arr
        else:
            row["npy_shape"] = "NOT FOUND"

        rows.append(row)

df = pd.DataFrame(rows)
print(df.to_string(index=False))

[1/28] Balboloop...
[2/28] Biswifo...
[3/28] Bomatog...
[4/28] Boochuto...
[5/28] Boumorim...
[6/28] Bovutou...
[7/28] Cadotueg...
[8/28] Detodu...
[9/28] Diecudey...
[10/28] Diepami...
[11/28] Diequipi...
[12/28] Dithigog...
[13/28] Dublafer...
[14/28] Dujomal...
[15/28] Elagieg...
[16/28] Golotag...
[17/28] Grequafie...
[18/28] Gueshifa...
[19/28] Kuquelok...
[20/28] Oduskueb...
[21/28] Quetode...
[22/28] Runusath...
[23/28] Sepigoo...
[24/28] Stonscuetof...
[25/28] Suquepog...
[26/28] Tercippun...
[27/28] Tiepolem...
[28/28] Tisupey...
 patient_id             hdf5_shape hdf5_dtype              npy_shape
  Balboloop (256, 256, 140, 20, 3)      int16 (20, 3, 140, 178, 256)
    Biswifo (256, 256, 120, 20, 3)      int16 (20, 3, 120, 204, 256)
    Bomatog (256, 256, 140, 20, 3)      int16 (20, 3, 140, 204, 256)
   Boochuto (256, 256, 120, 20, 3)      int16 (20, 3, 120, 192, 256)
   Boumorim (256, 256, 120, 20, 3)      int16 (20, 3, 120, 178, 256)
    Bovutou (256, 256, 140, 20, 3)      i

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

PATIENT_DATA_DIR = pc.working_dir / "patient_data"
VZ_FLOW_TAG = 5

from tqdm import tqdm

corr_vel_dir = pc.repository_root / "corrected_velocities"

def get_orig_direction(pid):
    catalog_path = PATIENT_DATA_DIR / pid / f"dicom_catalog_{pid}.csv"
    if not catalog_path.exists():
        return "UNKNOWN"
    catalog = pd.read_csv(catalog_path)
    vz_cat = catalog[catalog["tag_0x0043_0x1030"] == VZ_FLOW_TAG].copy()
    if len(vz_cat) == 0:
        return "UNKNOWN"
    vz_cat["time_index"] = (vz_cat["instancenumber"] - 1) % vz_cat["cardiacnumberofimages"]
    vz_cat["slice_index"] = (vz_cat["instancenumber"] - 1) // vz_cat["cardiacnumberofimages"]
    vz_cat["z"] = vz_cat["imagepositionpatient"].apply(lambda x: np.array(eval(x))[2])
    t0 = vz_cat[vz_cat["time_index"] == 0].sort_values("slice_index")
    z_diff = np.diff(t0["z"].values)
    if np.sum(z_diff > 0) > np.sum(z_diff < 0):
        return "I_to_S"
    elif np.sum(z_diff < 0) > np.sum(z_diff > 0):
        return "S_to_I"
    return "AMBIGUOUS"

out_dir = Path("evan_prediction_slices")
out_dir.mkdir(exist_ok=True)

comp_labels = ["v_x", "v_y", "v_z"]
t_idx = 0

with h5py.File(hdf5_path, "r") as f:
    for pi, pid in enumerate(test_patients):
        orig_dir = get_orig_direction(pid)
        flip_z = orig_dir == "S_to_I"
        print(f"[{pi+1}/{len(test_patients)}] {pid} ({orig_dir}){' [flipping z]' if flip_z else ''}...", flush=True)

        # HDF5: (X, Y, Z, T, 3)
        hdf5_vol = f[pid][:]
        if flip_z:
            hdf5_vol = hdf5_vol[:, :, ::-1, :, :]
            hdf5_vol[:, :, :, :, 2] = -hdf5_vol[:, :, :, :, 2]

        npy_path = corr_vel_dir / f"{pid}.npy"
        if not npy_path.exists():
            print(f"  npy not found, skipping")
            continue
        # NPY: (T, 3, Z, Y, X) — always I_to_S
        npy_arr = np.load(npy_path)

        n_z = hdf5_vol.shape[2]
        pid_dir = out_dir / pid
        pid_dir.mkdir(exist_ok=True)

        for z in range(0, n_z, 4):
            fig, axes = plt.subplots(3, 2, figsize=(8, 12))
            for c in range(3):
                hdf5_slice = hdf5_vol[:, :, z, t_idx, c]
                axes[c, 0].imshow(hdf5_slice, cmap="RdBu_r", origin="lower", vmin=-200, vmax=200)
                axes[c, 0].set_title(f"HDF5 {comp_labels[c]}")
                axes[c, 0].axis("off")

                npy_slice = npy_arr[t_idx, c, z, :, :]
                axes[c, 1].imshow(npy_slice, cmap="RdBu_r", origin="lower", vmin=-200, vmax=200)
                axes[c, 1].set_title(f"NPY {comp_labels[c]}")
                axes[c, 1].axis("off")

            dir_label = f"{orig_dir}, flipped" if flip_z else orig_dir
            fig.suptitle(f"{pid} — z={z}, t={t_idx} ({dir_label})", fontsize=14)
            fig.tight_layout()
            fig.savefig(pid_dir / f"z{z:03d}.png", dpi=100)
            plt.close(fig)

        del hdf5_vol, npy_arr

print("Done.")

[1/28] Balboloop (I_to_S)...
[2/28] Biswifo (S_to_I) [flipping z]...
[3/28] Bomatog (S_to_I) [flipping z]...
[4/28] Boochuto (S_to_I) [flipping z]...
[5/28] Boumorim (S_to_I) [flipping z]...
[6/28] Bovutou (I_to_S)...
[7/28] Cadotueg (S_to_I) [flipping z]...
[8/28] Detodu (I_to_S)...
[9/28] Diecudey (I_to_S)...
[10/28] Diepami (S_to_I) [flipping z]...
[11/28] Diequipi (S_to_I) [flipping z]...
[12/28] Dithigog (S_to_I) [flipping z]...
[13/28] Dublafer (I_to_S)...
[14/28] Dujomal (S_to_I) [flipping z]...
[15/28] Elagieg (I_to_S)...
[16/28] Golotag (S_to_I) [flipping z]...
[17/28] Grequafie (I_to_S)...
[18/28] Gueshifa (I_to_S)...
[19/28] Kuquelok (S_to_I) [flipping z]...
[20/28] Oduskueb (S_to_I) [flipping z]...
[21/28] Quetode (S_to_I) [flipping z]...
[22/28] Runusath (S_to_I) [flipping z]...
[23/28] Sepigoo (S_to_I) [flipping z]...
[24/28] Stonscuetof (S_to_I) [flipping z]...
[25/28] Suquepog (I_to_S)...
[26/28] Tercippun (S_to_I) [flipping z]...
[27/28] Tiepolem (S_to_I) [flipping z].

In [5]:
import nibabel as nib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import pydicom

PATIENT_DATA_DIR = pc.working_dir / "patient_data"
VZ_FLOW_TAG = 5

pid = "Balboloop"
t_idx = 0

# --- 1. Load DICOM vz pixel data for one slice (native DICOM order) ---
catalog_path = PATIENT_DATA_DIR / pid / f"dicom_catalog_{pid}.csv"
catalog = pd.read_csv(catalog_path)
vz_cat = catalog[catalog["tag_0x0043_0x1030"] == VZ_FLOW_TAG].copy()
vz_cat["time_index"] = (vz_cat["instancenumber"] - 1) % vz_cat["cardiacnumberofimages"]
vz_cat["slice_index"] = (vz_cat["instancenumber"] - 1) // vz_cat["cardiacnumberofimages"]
vz_cat["z"] = vz_cat["imagepositionpatient"].apply(lambda x: np.array(eval(x))[2])

vz_t0 = vz_cat[vz_cat["time_index"] == 0].sort_values("z", ascending=True).reset_index(drop=True)
print(f"Number of vz slices at t=0: {len(vz_t0)}")

mid_idx = len(vz_t0) // 2
dcm_path = vz_t0.iloc[mid_idx]["filepath"]
dcm = pydicom.dcmread(dcm_path)
dicom_pixels = dcm.pixel_array  # (Rows, Cols) — native DICOM pixel order
slope = float(getattr(dcm, "RescaleSlope", 1))
intercept = float(getattr(dcm, "RescaleIntercept", 0))
dicom_values = dicom_pixels.astype(np.float32) * slope + intercept
print(f"DICOM slice {mid_idx}: shape={dicom_pixels.shape}, slope={slope}, intercept={intercept}")

# --- 2. Load HDF5 vz at the same z-slice ---
with h5py.File(hdf5_path, "r") as f:
    hdf5_full = f[pid]  # (D0, D1, D2, T, 3)
    print(f"HDF5 shape: {hdf5_full.shape}")
    hdf5_vz_slice = hdf5_full[:, :, mid_idx, t_idx, 2].astype(np.float32)  # (D0, D1)
    print(f"HDF5 vz slice shape: {hdf5_vz_slice.shape}")

# --- 3. Load NIfTI vz at the same z-slice ---
uncorr_vz_path = (
    PATIENT_DATA_DIR / pid / "nifti"
    / f"4d_flow_vz_{pid}_per_timepoint_full_fov"
    / f"4d_flow_vz_{pid}_frame_{t_idx:02d}.nii.gz"
)
nii = nib.load(str(uncorr_vz_path))
nii_data = nii.get_fdata(dtype=np.float32)  # (i, j, k)
print(f"NIfTI shape: {nii_data.shape}, affine:\n{nii.affine}")
nii_vz_slice = nii_data[:, :, mid_idx]  # (i, j)
print(f"NIfTI vz slice shape: {nii_vz_slice.shape}")

# --- 4. Plot all three side by side with various orientations ---
out_dir = Path("evan_prediction_slices") / "vz_sign_check"
out_dir.mkdir(parents=True, exist_ok=True)

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
vmax = 200

axes[0, 0].imshow(dicom_values, cmap="RdBu_r", vmin=-vmax, vmax=vmax)
axes[0, 0].set_title(f"DICOM pixels (Rows×Cols)\n{dicom_values.shape}")

axes[0, 1].imshow(hdf5_vz_slice, cmap="RdBu_r", vmin=-vmax, vmax=vmax)
axes[0, 1].set_title(f"HDF5 [:,:,z,t,2]\n{hdf5_vz_slice.shape}")

axes[0, 2].imshow(nii_vz_slice, cmap="RdBu_r", vmin=-vmax, vmax=vmax)
axes[0, 2].set_title(f"NIfTI [:,:,z]\n{nii_vz_slice.shape}")

axes[1, 0].imshow(dicom_values, cmap="RdBu_r", origin="lower", vmin=-vmax, vmax=vmax)
axes[1, 0].set_title("DICOM (origin=lower)")

axes[1, 1].imshow(hdf5_vz_slice.T, cmap="RdBu_r", origin="lower", vmin=-vmax, vmax=vmax)
axes[1, 1].set_title(f"HDF5 .T\n{hdf5_vz_slice.T.shape}")

axes[1, 2].imshow(nii_vz_slice.T, cmap="RdBu_r", origin="lower", vmin=-vmax, vmax=vmax)
axes[1, 2].set_title(f"NIfTI .T\n{nii_vz_slice.T.shape}")

for ax in axes.flat:
    ax.axis("off")

fig.suptitle(f"{pid} — z={mid_idx}, t={t_idx} — axis orientation check", fontsize=14)
fig.tight_layout()
fig.savefig(out_dir / f"{pid}_axis_check.png", dpi=120)
plt.close(fig)
print(f"Saved to {out_dir / f'{pid}_axis_check.png'}")

Number of vz slices at t=0: 140
DICOM slice 70: shape=(256, 256), slope=1.0, intercept=0.0
HDF5 shape: (256, 256, 140, 20, 3)
HDF5 vz slice shape: (256, 256)
NIfTI shape: (256, 256, 140), affine:
[[ -1.40625083   0.           0.         158.86599731]
 [  0.          -1.40625083   0.         148.21499634]
 [  0.           0.           1.79987395 -89.54589844]
 [  0.           0.           0.           1.        ]]
NIfTI vz slice shape: (256, 256)
Saved to evan_prediction_slices/vz_sign_check/Balboloop_axis_check.png
